In [2]:
import pandas as pd
import numpy as np
import os
import sys
from notebook.services.config import ConfigManager
cm = ConfigManager().update('notebook', {'limit_output': 1000})
pd.options.display.max_rows=999
pd.options.display.max_columns=999
pd.options.display.max_seq_items=999
%matplotlib widget
import matplotlib.pyplot as plt
from edw_functions import *
sys.path.append('/home/wolfgang/repos/sleep_research_io/')
from sleep_research_functions import *
%load_ext autoreload
%autoreload 2
import re
from datetime import datetime, timedelta
from apnea_detection_functions import self_similarity_indices
import pdb
from sleep_analysis_functions import *

from airgo_features import compute_airgo_features

from data_summary_extraction import *

In [8]:
# apnea_preds_cols = ['apnea_binary',
#  'apnea_end',
#  'apnea_pred_any_a3',
#  'apnea_pred_ro_a3',
#  'apnea_pred_ro_a3_ss',
#  'apnea_pred_rsr_a3',
#  'apnea_pred_rsr_a3_ss',
#  'apnea_pred_va_a3',
#  'apnea_prob_ro_a3',
#  'apnea_prob_rsr_a3']
# stages_cols = ['stage_pred_activity10sec',
#  'stage_pred_amendsumeffort',
#  'stage_pred_comb_breath_activity_1']
# airgo_cols = ['band', 'band_unscaled', 'accx', 'accy', 'accz', 'movavg_0_5s',
#        'movavg_1_2s', 'movavg_1min', 'deriv1', 'ventilation0',
#        'ventilation_3s', 'ventilation_8s', 'ventilation_10s',
#        'ventilation_12s', 'ventilation_15s', 'ventilation_30s',
#        'ventilation_60s', 'ventilation_5min', 'ventilation_10min',
#        'ventilation_10s_smooth', 'hypo_10_60', 'movmedian_5min',
#        'movmedian_10min', 'movstd_8s', 'movstd_12s', 'movstd_15s',
#        'movstd_10s', 'movstd_30s', 'movstd_60s', 'movstd_5min', 'movstd_10min',
#        'katz_fd_10s_smoothed', 'katz_fd_30s_smoothed', 'katz_fd_45s_smoothed',
#        'katz_fd_60s_smoothed', 'sample_entropy_10s_smoothed',
#        'sample_entropy_30s_smoothed', 'sample_entropy_60s_smoothed',
#        'movavg_0_5s_unscaled', 'peaks', 'movstd_1min_unscaled',
#        'movstd_30sec_unscaled', 'rr_10s', 'rr_60s', 'rr_10s_smooth',
#        'movmedian_1min', 'movmedian_30sec', 'IBI', 'IBI_mean_5min',
#        'IBI_std_5min', 'ventilation_5min_deriv', 'troughs', 'inht', 'exht',
#        'inht_exht_ratio', 'inht_cycle_ratio', 'inht_exht_ratio_10sec',
#        'inht_cycle_ratio_10sec', 'TVpB', 'TVpB_regularity',
#        'TVpB_regularity_10sec', 'accx_1sec', 'accy_1sec', 'accz_1sec',
#        'acc_ai_1sec', 'acc_ai_10sec', 'acc_enmo', 'acc_enmo_1sec',
#        'acc_enmo_10sec', 'position_cluster', 'ventilation_cvar_30sec',
#        'IBI_cvar_30sec', 'instability_index_30sec', 'ventilation_cvar_1min',
#        'IBI_cvar_1min', 'instability_index_1min', 'ventilation_cvar_2min',
#        'IBI_cvar_2min', 'instability_index_2min', 'ventilation_cvar_5min',
#        'IBI_cvar_5min', 'instability_index_5min']

# airgo_cols.extend(apnea_preds_cols)
# airgo_cols.extend(stages_cols)
# airgo_cols = [x.lower() for x in airgo_cols]

In [9]:
dir_input = f'/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_step3'
dir_input = f'/media/mad3/Projects/ICU_SLEEP_STUDY/Sleep_And_Breathing/icu_files_step3_nohrv'


In [10]:
# clinical table, where data gets added:
table_path = '/home/wolfgang/repos/Undiagnosed_Apnea/icu_sleep_analysis_table.csv'
table = pd.read_csv(table_path)
table.rename({'Study ID': 'study_id'}, inplace=True, axis=1)
# remove dummy biosignals columns:
table = table[table.columns[:21]]

In [11]:
table;

In [12]:
files = os.listdir(dir_input)
files.sort()
print(len(files))

130


In [13]:
summary_subjects = pd.DataFrame()
summary_days = pd.DataFrame()

overwrite = True

In [47]:
print('cpc statistic deactivated currently as data does not contain HRV.')

for file in ['icusleep_126.h5']: # files:
    
#     try:
    if 1:
        filepath = os.path.join(dir_input, file)
        study_id = re.search('\d\d\d', file)[0]
        
        print(study_id)

        data, hdr, fs = read_in_routine(filepath)
        data['edw_bp_map'] = (data.edw_bp_systolic + 2*data.edw_bp_diastolic) / 3

        if data.band.dropna().shape[0] == 0:
            print(f'No valid AirGo data any more, skip {file}')
            continue
        timerange = 'all'
        statistics_to_extract = 'default'

        summary_subject, summary_days_subject = extract_summary_tables_for_subject(data, timerange, statistics_to_extract, study_id = study_id, clinical_table = table)

        summary_subjects = pd.concat([summary_subjects, summary_subject], axis=0, sort=False)
        summary_days = pd.concat([summary_days, summary_days_subject], axis=0, sort=False)
        
#     except Exception as e:
#         print(file)
#         print(e)
#         continue


cpc statistic deactivated currently as data does not contain HRV.
126


In [ ]:
# 042
# icusleep_042.h5
# index 0 is out of bounds for axis 0 with size 0

# 126
# icusleep_126.h5

In [28]:
# summary_days.loc[:, ['cam_s_morning', 'cam_s_evening', 'day_cat', 'day_no', 'timerange']]

In [58]:
# summary_subjects = summary_subjects.sort_values(by='study_id')
# summary_days = summary_days.sort_values(by=['study_id', 'day_no'])

In [63]:
summary_subjects.to_csv('summary_subjects.csv', index=False)
summary_days.to_csv('summary_days.csv', index=False)

In [29]:
summary_subjects2 = summary_subjects.copy()
summary_days2 = summary_days.copy()

In [26]:
len(summary_days.study_id.unique())

128

In [27]:
len(summary_subjects.study_id.unique())

128

In [162]:
apnea_name = 'apnea_pred_va_a3'  # name of Apnea info column
hours_sleep = 'stage_pred_amendsumeffort'  # name of sleep stage column, or int/float for manual setting.
hypoxia_name = 'hypoxic_area' + apnea_name.replace('apnea_pred', '')


In [205]:
data, hypoxia_burden = compute_hypoxia_burden(data, fs, apnea_name = apnea_name, 
		    	hypoxia_name=hypoxia_name, hours_sleep = hours_sleep)

In [109]:
data.columns

Index(['acc_ai_10sec', 'acc_ai_1sec', 'acc_enmo', 'acc_enmo_10sec',
       'acc_enmo_1sec', 'accx', 'accx_1sec', 'accy', 'accy_1sec', 'accz',
       'accz_1sec', 'antipsychotics_sum', 'apnea_binary', 'apnea_end',
       'apnea_pred_any_a3', 'apnea_pred_ro_a3', 'apnea_pred_ro_a3_ss',
       'apnea_pred_rsr_a3', 'apnea_pred_rsr_a3_ss', 'apnea_pred_va_a3',
       'apnea_prob_ro_a3', 'apnea_prob_rsr_a3', 'art1d', 'art1s', 'band',
       'band_unscaled', 'benzos_sum', 'cam_s', 'cpc_cntr_window', 'cpc_hfc',
       'cpc_hfc_lfc_ratio', 'cpc_lfc', 'deriv1', 'dex_studydrug', 'ecg_signal',
       'edw_bp_diastolic', 'edw_bp_systolic', 'edw_pulse',
       'edw_pulse_oximetry', 'edw_respirations', 'edw_temperature',
       'edw_urine_output', 'envelope_lo', 'envelope_up', 'exht',
       'hco3_arterial', 'hco3_venous', 'hr', 'hrv_ac', 'hrv_apen',
       'hrv_avgsqi', 'hrv_btsdet', 'hrv_dc', 'hrv_fdflag', 'hrv_hf', 'hrv_iqr',
       'hrv_lf', 'hrv_lfhf', 'hrv_nnkurt', 'hrv_nnmean', 'hrv_nnmedian',
 

In [84]:
statistics_to_extract

{'hr': ['mean', 'std', 'median', 'q25', 'q75'],
 'spo2': ['mean', 'std', 'median', 'q25', 'q75'],
 'bp_sys': ['mean', 'std', 'median', 'q25', 'q75'],
 'bp_dia': ['mean', 'std', 'median', 'q25', 'q75'],
 'bp_map': ['mean', 'std', 'median', 'q25', 'q75'],
 'cpc_hflf_ratio': ['mean', 'std', 'median', 'q25', 'q75'],
 'hco3': ['mean', 'min', 'max'],
 'pco2': ['mean', 'min', 'max'],
 'po2': ['mean', 'min', 'max'],
 'ph': ['mean', 'min', 'max'],
 'sofa': ['mean', 'min', 'max'],
 'cam_s': ['mean', 'min', 'max'],
 'apnea_pred_...': ['index'],
 'hypoxic_burden': ['hypoxic_burden_sum'],
 'self_similarity': ['mean'],
 'opioids_sum': ['sum'],
 'benzos_sum': ['sum'],
 'antipsychotics_sum': ['sum'],
 'airgo_available': ['hours'],
 'sleep_hours': ['hours'],
 'stages_W': ['perc'],
 'stages_R': ['perc'],
 'stages_N1': ['perc'],
 'stages_N2': ['perc'],
 'stages_N3': ['perc']}

In [85]:
timerange_dict

{'d1': '2019-03-12 08:00:00 - 2019-03-12 20:00:00',
 'n1': '2019-03-12 20:00:00 - 2019-03-13 08:00:00',
 'f1': '2019-03-12 08:00:00 - 2019-03-13 08:00:00',
 'd2': '2019-03-13 08:00:00 - 2019-03-13 20:00:00',
 'n2': '2019-03-13 20:00:00 - 2019-03-14 08:00:00',
 'f2': '2019-03-13 08:00:00 - 2019-03-14 08:00:00',
 'd3': '2019-03-14 08:00:00 - 2019-03-14 20:00:00',
 'n3': '2019-03-14 20:00:00 - 2019-03-15 08:00:00',
 'f3': '2019-03-14 08:00:00 - 2019-03-15 08:00:00'}